<a href="https://colab.research.google.com/github/SergiSama/UIC-CRB1-2026-2027/blob/main/m3_integration_pid_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 · Notebook 2 — Integration & the PID Controller
### Computing, Robotics & Bionics · companion to A. Géron and to the Control Theory unit

A **self-contained** integral-calculus notebook that connects the calculus refresher to your **Control
Theory** unit. Integration is *accumulation* (area under a curve); its clearest engineering payoff is the
**integral term of a PID controller** — the "I" that accumulates error to remove steady-state offset. We
build Riemann sums, see the Fundamental Theorem numerically, measure how fast each integration rule
converges, integrate a physiological flow signal into a volume (and watch sensor bias make it drift), learn
that every simulation is itself an integral (Euler's method, with the same step-size law as gradient descent),
and build a PID controller from scratch: P vs PI, tuning predicted from a quadratic equation, integral windup,
and the D term on a prosthetic elbow.

Run in **Google Colab** (*Runtime → Run all*). Everything is offline.

**Contents**
1. The integral as accumulation: Riemann sums
2. The Fundamental Theorem (numerically)
3. Numerical integration: the trapezoidal rule — and how fast each rule converges (Simpson)
4. Accumulation in physiology: flow → volume — and why integrated signals drift
5. Simulating a system is integration: Euler's method and its step-size law
6. The PID integral: P vs PI control from scratch — and why the offset must vanish
7. Tuning the integral gain: the closed loop as a second-order ODE; integral windup
8. The D term: damping a prosthetic elbow, derivative kick, and noise

Most sections end with an **Exercise**; run the **Solution** cell to check.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
print("ready")

---
## 1 · The integral as accumulation: Riemann sums

The definite integral is the limit of a sum of thin rectangles — the area under the curve. We visualise
the rectangles for `f(x)=x^2` on `[0,1]` and watch the sum approach the exact area `1/3` as the number of
rectangles grows.

In [ ]:
f = lambda x: x**2
def riemann(f, a, b, N, kind="mid"):
    edges = np.linspace(a, b, N+1); dx = (b-a)/N
    if kind == "left":  xs = edges[:-1]
    elif kind == "right": xs = edges[1:]
    else: xs = (edges[:-1] + edges[1:]) / 2     # midpoint
    return np.sum(f(xs)) * dx

for N in [4, 10, 50, 1000]:
    print(f"N={N:<5} midpoint sum = {riemann(f,0,1,N):.6f}  (exact 1/3 = {1/3:.6f})")

In [ ]:
# Visualise the rectangles (N = 8, left rule)
N = 8; a, b = 0, 1; edges = np.linspace(a, b, N+1); dx = (b-a)/N
xs = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(xs, f(xs), color="navy", lw=2, label="f(x)=x^2")
ax.bar(edges[:-1], f(edges[:-1]), width=dx, align="edge",
       alpha=0.4, color="steelblue", edgecolor="white", label="rectangles")
ax.set_title(f"Riemann sum, N={N}: area approximates the integral")
ax.legend(); ax.grid(alpha=0.3); plt.show()

**Exercise 1.** During a 10-minute exercise test a patient's oxygen-consumption rate is modelled as `vo2(t) = 0.3 + 0.05*t` L/min (`t` in minutes). Use a midpoint Riemann sum (reuse `riemann` above) with `N=100` sub-intervals to estimate the total oxygen consumed (litres) over `[0, 10]`, and compare it to the exact value from the antiderivative.

> 🤖 *Gemini tip:* "Given a Python function of time representing a physiological rate, show me how to use a midpoint Riemann sum to estimate the total accumulated quantity over an interval, and how to check it against the exact antiderivative."

In [ ]:
vo2 = lambda t: 0.3 + 0.05*t

# Your code here

---
## 2 · The Fundamental Theorem (numerically)

Integration and differentiation are inverses. If we **accumulate** a rate we recover the quantity, and if
we **differentiate** that accumulation we recover the rate. We check both numerically on `f(x)=cos(x)`,
whose antiderivative is `sin(x)`.

In [ ]:
x = np.linspace(0, 2*np.pi, 400); dx = x[1]-x[0]
rate = np.cos(x)
# accumulate the rate (cumulative trapezoid) -> should match sin(x)
accum = np.concatenate([[0], np.cumsum((rate[1:] + rate[:-1]) / 2 * dx)])
# differentiate the accumulation back -> should match cos(x)
back = np.gradient(accum, dx)
print("max |accum - sin(x)| :", np.max(np.abs(accum - np.sin(x))))
print("max |d(accum) - cos| :", np.max(np.abs(back - rate)))
fig, ax = plt.subplots(figsize=(7,3))
ax.plot(x, rate, label="rate  cos(x)")
ax.plot(x, accum, label="accumulation -> sin(x)")
ax.legend(); ax.grid(alpha=0.3); ax.set_title("Accumulating a rate recovers the quantity"); plt.show()

**Exercise 2.** Verify the Fundamental Theorem on a linearly ramping oxygen-uptake rate `rate(t) = 2*t` over `t` in `[0, 5]`: accumulate it numerically (cumulative trapezoid) and confirm it matches the exact antiderivative `t**2`, then differentiate the accumulation back with `np.gradient` and confirm it recovers `rate(t)`.

> 🤖 *Gemini tip:* "Show me how to numerically verify the Fundamental Theorem of Calculus for a given rate function: accumulate it with a cumulative trapezoid rule, compare it to the known antiderivative, then differentiate the accumulation back with np.gradient."

In [ ]:
tt = np.linspace(0, 5, 300); dtt = tt[1]-tt[0]
rate2 = 2*tt

# Your code here

---
## 3 · Numerical integration: the trapezoidal rule

When `f` is data, approximate the area with trapezoids. We implement the rule by hand (it is just a
weighted Riemann sum) and check it against known integrals.

In [ ]:
def trapezoid(y, dx):
    return (0.5*y[0] + y[1:-1].sum() + 0.5*y[-1]) * dx

# integral of x^2 on [0,1] = 1/3
xg = np.linspace(0, 1, 101); dx = xg[1]-xg[0]
print("trapezoid x^2 :", round(trapezoid(xg**2, dx), 6), " exact 1/3 =", round(1/3, 6))
# integral of sin on [0, pi] = 2
xs = np.linspace(0, np.pi, 201); dxs = xs[1]-xs[0]
print("trapezoid sin :", round(trapezoid(np.sin(xs), dxs), 6), " exact = 2")

**Exercise 3.** Use the trapezoidal rule to approximate the integral of `e^x` on `[0, 1]` with 100
intervals, and compare to the exact value `e - 1`.

> 🤖 *Gemini tip:* "Show me how to implement the trapezoidal rule for numerical integration in NumPy from an array of function values, and how to check it against a known exact integral."

In [ ]:
xe = np.linspace(0, 1, 101); dxe = xe[1]-xe[0]

# Your code here

### 3.1 · How fast the error shrinks — and Simpson's rule

Every rule replaces $f$ on a strip of width $h$ by something simpler: a flat top taken at the left edge
(left rule), a flat top taken at the centre (midpoint), a straight line through both ends (trapezoid), or
a parabola through three points (**Simpson**). Taylor (Notebook 1, Section 5) predicts how the total error
scales with $h$:

| rule | exact for | total error |
|---|---|---|
| left / right | constants | $O(h)$ |
| midpoint, trapezoid | straight lines | $O(h^2)$ |
| Simpson | cubics (a symmetry bonus) | $O(h^4)$ |

Simpson's rule needs an even number of strips $N$:
$$\int_a^b f\,dt\;\approx\;\frac h3\Big[f_0+4f_1+2f_2+4f_3+\dots+2f_{N-2}+4f_{N-1}+f_N\Big].$$

An error $O(h^p)$ means that **halving $h$ divides the error by $2^p$** — by 2, 4 and 16 here. That is the
same bookkeeping that ranked the forward and central differences in Notebook 1, now for integrals.

In [ ]:
def simpson(y, dx):
    """Simpson's rule on equally spaced samples y (len(y)-1 must be even)."""
    assert (len(y) - 1) % 2 == 0, "Simpson needs an even number of intervals"
    return dx / 3 * (y[0] + y[-1] + 4 * y[1:-1:2].sum() + 2 * y[2:-1:2].sum())


exact_e = np.e - 1
Ns = np.array([4, 8, 16, 32, 64, 128])
errs_rule = {"left": [], "trapezoid": [], "Simpson": []}
for N in Ns:
    xq = np.linspace(0, 1, N + 1); hq = 1 / N; yq = np.exp(xq)
    errs_rule["left"].append(abs(yq[:-1].sum() * hq - exact_e))
    errs_rule["trapezoid"].append(abs(trapezoid(yq, hq) - exact_e))
    errs_rule["Simpson"].append(abs(simpson(yq, hq) - exact_e))

print("integral of e^x on [0,1]; error ratio each time h is halved")
for name, e in errs_rule.items():
    e = np.array(e)
    slope = np.polyfit(np.log(1 / Ns), np.log(e), 1)[0]
    print(f"  {name:9s}  ratios {np.round(e[:-1] / e[1:], 1)}  -> measured order {slope:.2f}")

fig, ax = plt.subplots(figsize=(6.2, 3.9))
for (name, e), col in zip(errs_rule.items(), ["orange", "teal", "crimson"]):
    ax.loglog(1 / Ns, e, "o-", color=col, label=name)
ax.set_xlabel("strip width h"); ax.set_ylabel("|error|")
ax.set_title("Slopes 1, 2 and 4 on a log-log plot", fontsize=10)
ax.legend(); ax.grid(alpha=0.3, which="both"); plt.show()

**Exercise 4.** In pharmacokinetics the **area under the concentration curve** (AUC) measures total drug exposure. After
an oral dose, a patient's plasma concentration is modelled as $C(t)=10\,(e^{-0.2t}-e^{-1.5t})$ mg/L
($t$ in hours). (a) Compute the exact $\mathrm{AUC}_{0\text{–}24}$ from the antiderivative. (b) Suppose blood is
sampled every 4, 2, 1 and 0.5 hours; estimate the AUC from those samples with the trapezoidal rule and
with Simpson's rule. (c) Measure the order of each rule from your errors. Which sampling interval would
you need with the trapezoid to match Simpson's accuracy at hourly samples?

> 🤖 *Gemini tip:* "Explain how the area under a drug concentration curve (AUC) is computed from blood samples with the trapezoidal rule, and how to estimate the order of accuracy of a numerical integration rule from its errors."

In [ ]:
# Your code here

---
## 4 · Accumulation in physiology: flow → volume

Integrating a respiratory **flow** (litres/second) over time gives lung **volume** (litres) — the area
under the flow curve. We synthesise a few breaths of flow and accumulate it into the tidal-volume trace.

In [ ]:
fs = 100; T = 12.0                       # 12 s at 100 Hz
t = np.arange(0, T, 1/fs)
breath_rate = 0.25                       # Hz (15 breaths/min)
flow = 0.5 * np.sin(2*np.pi*breath_rate*t)   # L/s: + inhale, - exhale
flow += np.random.default_rng(0).normal(0, 0.01, t.shape)

dt = 1/fs
volume = np.concatenate([[0], np.cumsum((flow[1:]+flow[:-1])/2 * dt)])  # integrate flow

fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax[0].plot(t, flow, color="teal"); ax[0].axhline(0, color="gray", lw=0.6)
ax[0].set_ylabel("flow (L/s)"); ax[0].set_title("Respiratory flow")
ax[1].plot(t, volume, color="crimson"); ax[1].set_ylabel("volume (L)")
ax[1].set_xlabel("time (s)"); ax[1].set_title("Lung volume = integral of flow")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("tidal volume (peak-to-trough):", round(volume.max()-volume.min(), 3), "L")

**Exercise 5.** Using the `flow` and `volume` arrays above, compute (a) the average inspiratory flow rate (mean of the positive part of `flow`), (b) the tidal volume, and (c) the minute ventilation (tidal volume x breaths per minute) in L/min.

> 🤖 *Gemini tip:* "Given an array of respiratory flow measurements and its integral (volume), show me how to compute the average inspiratory flow and estimate minute ventilation from the tidal volume and breathing rate."

In [ ]:
# Your code here

### 4.1 · Integration accumulates errors: drift

The flow above was clean. A real flow sensor is not: it has a small **bias** (it reads $\beta$ L/s when
nothing flows) and **noise**. Integration adds up *everything*, errors included:

* a bias $\beta$ contributes $\int_0^t\beta\,d\tau=\beta\,t$ — the volume **drifts linearly**, forever;
* independent noise of size $\sigma$ per sample adds like a random walk: its contribution grows like
  $\sigma\sqrt{\Delta t\,t}$ — slower, but also without bound.

A bias of only $0.01$ L/s is invisible on the flow trace; after a minute it has added 0.6 L to the volume,
about one tidal volume. The standard fix uses physiology: over whole breaths, air in equals air out, so
the true mean flow is zero. Subtracting the measured mean flow before integrating removes the bias.

In [ ]:
dr_fs, dr_T = 100, 60.0
dr_t = np.arange(0, dr_T, 1 / dr_fs); dr_dt = 1 / dr_fs
dr_true = 0.5 * np.sin(2 * np.pi * 0.25 * dr_t)                  # 15 breaths/min, as above
dr_bias, dr_sigma = 0.01, 0.01
dr_meas = dr_true + dr_bias + np.random.default_rng(1).normal(0, dr_sigma, dr_t.shape)

cumtrap = lambda y, h: np.concatenate([[0], np.cumsum((y[1:] + y[:-1]) / 2 * h)])
vol_true = cumtrap(dr_true, dr_dt)
vol_raw = cumtrap(dr_meas, dr_dt)
vol_fix = cumtrap(dr_meas - dr_meas.mean(), dr_dt)               # remove the mean flow first

print(f"volume error after {dr_T:.0f} s: raw {vol_raw[-1] - vol_true[-1]:+.3f} L "
      f"(bias x t predicts {dr_bias * dr_T:+.3f} L) | after mean removal {vol_fix[-1] - vol_true[-1]:+.4f} L")

# The noise part alone: 300 sensors with no bias, to see the sqrt(t) spread
dr_walks = np.array([cumtrap(np.random.default_rng(k).normal(0, dr_sigma, dr_t.shape), dr_dt) for k in range(300)])

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(dr_t, vol_raw, color="crimson", lw=1, label="integrated raw flow")
ax[0].plot(dr_t, vol_fix, color="teal", lw=1, label="mean flow removed first")
ax[0].plot(dr_t, dr_bias * dr_t, "k--", lw=1, label="β·t")
ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("volume (L)")
ax[0].set_title("A 0.01 L/s bias becomes a 0.6 L drift in a minute", fontsize=10)
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].plot(dr_t, dr_walks.std(axis=0), color="navy", label="spread of 300 noisy integrals")
ax[1].plot(dr_t, dr_sigma * np.sqrt(dr_dt * dr_t), "k--", label="σ·√(Δt·t)")
ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("std of volume error (L)")
ax[1].set_title("Noise alone: a random walk that grows like √t", fontsize=10)
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Exercise 6.** A wrist-worn IMU is meant to track hand position by integrating acceleration **twice**
(acceleration → velocity → position). Simulate a wrist that is perfectly still for 10 s, sampled at
100 Hz, with an accelerometer bias of $0.02$ m/s² and noise of $0.05$ m/s². (a) Integrate twice with the
cumulative trapezoid and report the apparent velocity and position after 10 s. (b) Show that the bias
alone predicts a velocity of $\beta t$ and a position of $\tfrac12\beta t^2$, and compare. (c) Why is double
integration so much worse than single integration, and why do IMU-based trackers need an external
reference (a camera, or zero-velocity updates at every footstep)?

> 🤖 *Gemini tip:* "Explain why integrating accelerometer data twice to get position drifts quadratically with time, and what techniques such as zero-velocity updates are used to correct it."

In [ ]:
# Your code here

---
## 5 · Simulating a system is integration: Euler's method

A model such as $\dot x=-a\,x+b\,u$ tells you how fast $x$ **changes**, not what $x$ **is**. To get $x(t)$
you integrate the rate: $x(t)=x(0)+\int_0^t \dot x\,d\tau$. **Euler's method** does it with a left Riemann
sum, one strip at a time:
$$x_{k+1}=x_k+\Delta t\,f(x_k).$$
Every simulation in the rest of this notebook — the line `x += dt*(-a*x + b*u)` — is exactly this. Two
consequences follow from Section 3.1.

* **Accuracy.** Euler is a left rule, so its error is $O(\Delta t)$: halve the step, halve the error.
* **Stability.** For $\dot x=-\lambda x$ one step gives $x_{k+1}=(1-\lambda\Delta t)\,x_k$. That factor must
  satisfy $|1-\lambda\Delta t|<1$, so the simulation is stable only if
  $$\Delta t<\frac{2}{\lambda}.$$

That is **Notebook 1's learning-rate law** $\eta^*=2/\lambda_{\max}$, letter for letter. Gradient descent *is*
Euler's method applied to the "gradient flow" $\dot{\mathbf w}=-\nabla L(\mathbf w)$, with the learning rate
playing the time step. Take too big a step and either one explodes.

In [ ]:
# Drug elimination: dx/dt = -k x, x(0) = 1 (fraction of the dose left). Exact: exp(-k t).
eu_k, eu_T = 1.0, 6.0

def euler_decay(dt_e, T_e=eu_T):
    n_e = int(round(T_e / dt_e))
    xe = np.empty(n_e + 1); xe[0] = 1.0
    for j in range(n_e):
        xe[j + 1] = xe[j] + dt_e * (-eu_k * xe[j])      # one left-Riemann strip of the rate
    return np.arange(n_e + 1) * dt_e, xe

print("accuracy: error at t = 2 h as the step is halved")
prev = None
for dt_e in [0.2, 0.1, 0.05, 0.025]:
    te, xe = euler_decay(dt_e)
    err_e = abs(xe[np.argmin(abs(te - 2))] - np.exp(-2 * eu_k))
    print(f"  dt = {dt_e:5.3f}: error {err_e:.5f}" + (f"   ratio {prev / err_e:.2f}" if prev else ""))
    prev = err_e

print(f"stability: the law predicts dt < 2/k = {2 / eu_k:.1f}")
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
tf = np.linspace(0, eu_T, 300)
for dt_e, col in [(0.1, "teal"), (0.5, "seagreen"), (1.5, "orange")]:
    te, xe = euler_decay(dt_e)
    ax[0].plot(te, xe, "o-", ms=3, color=col, label=f"Euler, Δt = {dt_e}  (factor {1 - eu_k * dt_e:+.1f})")
ax[0].plot(tf, np.exp(-eu_k * tf), "k--", lw=1, label="exact e^(−kt)")
ax[0].set_title("Stable steps: small ones are accurate, big ones ring", fontsize=10)
for dt_e, col in [(1.9, "orange"), (2.1, "crimson")]:
    te, xe = euler_decay(dt_e, T_e=60)
    ax[1].plot(te, xe, "o-", ms=3, color=col, label=f"Δt = {dt_e}  (factor {1 - eu_k * dt_e:+.1f})")
ax[1].set_title("Either side of Δt = 2/k, over 60 h", fontsize=10)
for a_ in ax:
    a_.set_xlabel("time (h)"); a_.set_ylabel("fraction of dose left"); a_.grid(alpha=0.3); a_.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Exercise 7.** Feedback changes the speed of the loop, and with it the largest safe time step. Under P-only control
the plant $\dot x=-a x+b u$ with $u=K_p(r-x)$ becomes $\dot x=-(a+bK_p)\,x+bK_p r$.
(a) With $a=b=1$ and $K_p=2$, **predict** the largest stable Euler step. (b) Write
`simulate_dt(Kp, dt_s, T_s=30)` — the P-only loop of the next section with its own time step — and
**test** your prediction at $\Delta t$ just below and just above it. (c) The open-loop plant alone
tolerates $\Delta t<2$. Why does adding the controller shrink the safe step?

> 🤖 *Gemini tip:* "Explain how the stability limit of Euler's method for dx/dt = -lambda x depends on lambda, and why closing a feedback loop around a system can make an explicit simulation unstable at a step that was fine before."

In [ ]:
# Your code here

---
## 6 · The PID integral: P vs PI control from scratch

A controller drives a system to a **setpoint** by acting on the error `e = setpoint - output`. A
**proportional (P)** controller leaves a steady-state **offset** (it needs a non-zero error to push). The
**integral (I)** term accumulates that residual error — a running Riemann sum — and keeps raising the
output until the error reaches zero. Every simulation below is Euler's method from Section 5. We simulate a first-order biomedical system (e.g. regulating a drug
concentration or temperature) under P-only and PI control.

In [ ]:
# First-order plant: dx/dt = -a*x + b*u   (Euler integration)
a, b = 1.0, 1.0
setpoint = 1.0
dt, T = 0.02, 12.0
steps = int(T/dt); time = np.arange(steps)*dt

def simulate(Kp, Ki):
    x = 0.0; integral = 0.0; xs = []
    for _ in range(steps):
        e = setpoint - x
        integral += e * dt              # the integral term = accumulated error (Riemann sum)
        u = Kp*e + Ki*integral
        x += dt * (-a*x + b*u)          # advance the plant one step
        xs.append(x)
    return np.array(xs)

x_P  = simulate(Kp=2.0, Ki=0.0)         # proportional only
x_PI = simulate(Kp=2.0, Ki=1.5)         # proportional + integral

ss_offset = setpoint - x_P[-1]
print("P-only steady-state value:", round(x_P[-1], 3), "-> offset", round(ss_offset, 3))
print("PI steady-state value    :", round(x_PI[-1], 3), "-> offset", round(setpoint - x_PI[-1], 3))

fig, ax = plt.subplots(figsize=(8,4))
ax.axhline(setpoint, ls="--", color="gray", label="setpoint")
ax.plot(time, x_P,  color="orange", label="P only (leaves offset)")
ax.plot(time, x_PI, color="crimson", label="PI (offset removed)")
ax.set_xlabel("time (s)"); ax.set_ylabel("output"); ax.legend()
ax.set_title("The integral term erases the steady-state offset"); ax.grid(alpha=0.3); plt.show()

This is exactly the qualitative story of the Control Theory unit (§5), made quantitative: the "I" in
PID is the integral of the error, implemented as an accumulator updated every time step.

### 6.1 · Why the offset *must* vanish

"The integral keeps pushing until the error is zero" can be made exact with the Fundamental Theorem. The
integrator's state $I(t)=\int_0^t e\,d\tau$ obeys
$$\frac{dI}{dt}=e(t).$$
If the loop settles to *any* steady state, every signal stops changing — including $I$. So $dI/dt=0$, and
therefore $e=0$. The integral term never needs to know how big the load or the offset is; it simply
cannot sit still while any error remains.

The P-only loop has no such state. At steady state the plant needs a non-zero push, $u=a\,x/b$, and the
only source of push is $u=K_p e$, so $e=a\,x/(bK_p)\neq0$. In the PI loop the push comes from the **stored**
integral instead: $e=0$ and $u_{ss}=K_i I_{ss}$, so $I_{ss}=a\,r/(bK_i)$.

In [ ]:
def simulate_state(Kp, Ki):
    """Same loop as simulate(), but also return the integrator state."""
    x = 0.0; integral = 0.0; xs, Is = [], []
    for _ in range(steps):
        e = setpoint - x
        integral += e * dt
        u = Kp * e + Ki * integral
        x += dt * (-a * x + b * u)
        xs.append(x); Is.append(integral)
    return np.array(xs), np.array(Is)

x_st, I_st = simulate_state(2.0, 1.5)
print(f"final error        : {setpoint - x_st[-1]:.2e}")
print(f"final integral I   : {I_st[-1]:.4f}   predicted a*r/(b*Ki) = {a * setpoint / (b * 1.5):.4f}")
print(f"final push Ki * I  : {1.5 * I_st[-1]:.4f}   needed a*x/b = {a * setpoint / b:.4f}")

**Exercise 8.** Write a function `settling_time(x, tol=0.02)` that returns the first time (in seconds, using the `time` array above) at which the response `x` stays within `tol` (as a fraction of `setpoint`) of the setpoint for the rest of the simulation. Use it to compare the PI controller above (`Ki=1.5`) against a more aggressive one (`Ki=4.0`): which settles faster?

> 🤖 *Gemini tip:* "Given a simulated control response array and a setpoint, show me how to write a function that finds the settling time — the first time the response stays within a tolerance of the setpoint for good."

In [ ]:
# Your code here

---
## 7 · Tuning the integral gain

The integral gain `Ki` sets how fast accumulated error is corrected. Too small and the offset clears
slowly; too large and the response overshoots and oscillates (and the integral can "wind up", Section 7.2). We sweep a
few values.

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.axhline(setpoint, ls="--", color="gray", label="setpoint")
for Ki, col in [(0.0,"orange"), (0.5,"seagreen"), (1.5,"crimson"), (6.0,"purple")]:
    ax.plot(time, simulate(2.0, Ki), color=col, label=f"Ki={Ki}")
ax.set_xlabel("time (s)"); ax.set_ylabel("output"); ax.legend()
ax.set_title("Effect of the integral gain Ki"); ax.grid(alpha=0.3); plt.show()

**Exercise 9.** For the P-only controller, the steady-state value of a first-order plant is
`x_ss = b*Kp / (a + b*Kp) * setpoint`. Compute it for `Kp = 2` (with `a=b=1`) and confirm it matches the
simulated `x_P[-1]`. Then argue why increasing `Kp` shrinks — but never removes — the offset.

> 🤖 *Gemini tip:* "Given a first-order control system's steady-state formula, show me how to compute it in Python and compare it to a simulated steady-state value."

In [ ]:
# Your code here

### 7.1 · Predicting the tuning: the closed loop is a second-order ODE

Why does a larger $K_i$ first speed things up and then start to ring? Write the loop out and use the
Fundamental Theorem once more to get rid of the integral. With $\dot x=-a x+b u$, $u=K_p e+K_i\int e$ and
a constant setpoint ($\dot e=-\dot x$), differentiate the plant equation:
$$\ddot x=-a\dot x+b\big(K_p\dot e+K_i e\big)
\quad\Longrightarrow\quad
\ddot x+(a+bK_p)\,\dot x+bK_i\,x=bK_i\,r.$$
The PI loop is a **mass–spring–damper**: $K_i$ is the spring, $a+bK_p$ the damper. Trying $e\propto e^{st}$
gives the characteristic equation
$$s^2+(a+bK_p)\,s+bK_i=0,$$
and its roots govern the tail of the response, $e(t)\sim e^{-\sigma t}$ with $\sigma=-\max\operatorname{Re}(s)$:

* **real roots** ($K_i<K_i^*$): no ringing; the slow root sets the tail, and it speeds up as $K_i$ grows;
* **complex roots** ($K_i>K_i^*$): the tail rings, and its decay rate is stuck at $(a+bK_p)/2$.

The boundary, a double root, is **critical damping**:
$$K_i^*=\frac{(a+bK_p)^2}{4b}=2.25\quad\text{for }a=b=1,\ K_p=2.$$
So the calculus predicts the best $K_i$ before a single simulation runs. We test it by measuring the tail's
decay rate from each simulated response.

In [ ]:
def tail_rate(x, t0=4.0, t1=10.0):
    """Decay rate of the error's envelope, from a straight-line fit to log|e| on [t0, t1]."""
    env = np.maximum.accumulate(np.abs(setpoint - x)[::-1])[::-1]   # envelope: max of |e| from t onwards
    m = (time >= t0) & (time <= t1)
    return -np.polyfit(time[m], np.log(env[m]), 1)[0]

def predicted_rate(Kp, Ki):
    return -np.roots([1, a + b * Kp, b * Ki]).real.max()

Ki_crit = (a + b * 2.0) ** 2 / (4 * b)
print(f"predicted critical gain Ki* = {Ki_crit:.2f}")
Ki_scan = np.array([0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0])
rate_sim = np.array([tail_rate(simulate(2.0, k)) for k in Ki_scan])
Ki_fine = np.linspace(0.05, 8.5, 400)
for k, r_ in zip(Ki_scan, rate_sim):
    print(f"  Ki = {k:4.2f}: roots {np.round(np.roots([1, a + b * 2.0, b * k]), 3)}  predicted rate "
          f"{predicted_rate(2.0, k):.3f}  measured {r_:.3f}")

fig, ax = plt.subplots(figsize=(7, 3.9))
ax.plot(Ki_fine, [predicted_rate(2.0, k) for k in Ki_fine], color="navy", label="predicted from the roots")
ax.plot(Ki_scan, rate_sim, "o", color="crimson", label="measured from the simulation")
ax.axvline(Ki_crit, color="gray", ls="--"); ax.text(Ki_crit + 0.1, 0.2, "Ki* = 2.25\ncritical damping", fontsize=8)
ax.text(5.2, 1.3, "complex roots: rings,\nno faster", fontsize=8); ax.text(0.3, 1.1, "real roots:\nfaster as Ki grows", fontsize=8)
ax.set_xlabel("Ki"); ax.set_ylabel("decay rate of the error tail (1/s)")
ax.set_title("Tuning Ki, predicted by a quadratic equation", fontsize=10)
ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.3); plt.show()
print("This is why Ki = 4 settled faster than Ki = 1.5 in Exercise 8, and why pushing Ki further only adds ringing.")

Two honest footnotes. Right at $K_i^*$ the tail is $t\,e^{-1.5t}$ rather than a pure exponential, so a
straight-line fit reads it slow; just above $K_i^*$ the ringing is so slow (at $K_i=2.5$ its period is
$2\pi/0.5\approx13$ s) that a 6-second fitting window cannot see it, which is why the scan skips that
region. And the root picture predicts the *tail*, not the first
overshoot: the $K_p$ term acts on $\dot e$ and adds a zero to the response, which can produce a small
overshoot even with real roots. The next exercise shows where that matters.

**Exercise 10.** Now tune with $K_p=4$. (a) **Predict** the critical integral gain $K_i^*$ and the fastest tail decay rate
the loop can reach. (b) **Test**: measure `tail_rate` at $K_i=3$, at your $K_i^*$ and at $K_i=10$ (use a
`simulate`-style loop with the new $K_p$), and compare with `predicted_rate`. (c) Scan $K_i$ and find
where the peak overshoot first exceeds 1 %. Does overshoot start at $K_i^*$? Explain the difference using
the footnote above.

> 🤖 *Gemini tip:* "For a PI controller on a first-order plant, explain how the closed-loop characteristic equation s^2 + (a + b Kp) s + b Ki = 0 determines damping, and why a zero from the proportional term can cause overshoot even when both poles are real."

In [ ]:
# Your code here

### 7.2 · Integral windup: when the actuator saturates

Real actuators have limits. An infusion pump cannot run backwards ($u\ge0$) or faster than its maximum
rate. While $u$ is pinned at the limit the error persists, so the integral keeps growing — it **winds up**
— with no effect on the plant. When the output finally reaches the setpoint, the swollen integral keeps
pushing, and the output overshoots until the excess area has been paid back with error of the opposite
sign. (Area again: the overshoot is the integral *repaying* what it stored.)

The simplest cure is **conditional integration**: stop integrating while the actuator is saturated.

In [ ]:
def simulate_sat(Kp, Ki, u_max, antiwindup=False):
    x = 0.0; integral = 0.0; xs, us, Is = [], [], []
    for _ in range(steps):
        e = setpoint - x
        trial = integral + e * dt
        u_raw = Kp * e + Ki * trial
        u = min(max(u_raw, 0.0), u_max)          # the pump's physical limits
        if not (antiwindup and u != u_raw):      # conditional integration: freeze I while saturated
            integral = trial
        x += dt * (-a * x + b * u)
        xs.append(x); us.append(u); Is.append(integral)
    return np.array(xs), np.array(us), np.array(Is)

u_lim = 1.2                                       # the steady state needs u = 1, so there is little headroom
x_w, u_w, I_w = simulate_sat(2.0, 4.0, u_lim)
x_aw, u_aw, I_aw = simulate_sat(2.0, 4.0, u_lim, antiwindup=True)
x_free = simulate(2.0, 4.0)
print(f"peak output: unlimited pump {x_free.max():.3f} | limited pump {x_w.max():.3f} | limited + anti-windup {x_aw.max():.3f}")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].axhline(setpoint, ls="--", color="gray")
for xv, col, lab in [(x_free, "gray", "no limit"), (x_w, "crimson", "limit, plain PI"), (x_aw, "teal", "limit + anti-windup")]:
    ax[0].plot(time, xv, color=col, label=lab)
ax[0].set_title("Output", fontsize=10); ax[0].legend(fontsize=8)
# (the small jitter in the teal pump command near t = 0.5 s is the integrator switching on and off at the limit)
ax[1].plot(time, u_w, color="crimson"); ax[1].plot(time, u_aw, color="teal"); ax[1].axhline(u_lim, ls=":", color="k")
ax[1].set_title(f"Pump command (limit {u_lim})", fontsize=10)
ax[2].plot(time, I_w, color="crimson", label="winds up"); ax[2].plot(time, I_aw, color="teal", label="frozen while saturated")
ax[2].axhline(a * setpoint / (b * 4.0), ls=":", color="k"); ax[2].set_title("Integrator state I", fontsize=10); ax[2].legend(fontsize=8)
for a_ in ax: a_.set_xlim(0, 6); a_.set_xlabel("time (s)"); a_.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Exercise 11.** A second common anti-windup scheme is **clamping**: after each update, limit the integrator itself to the
range it could ever usefully need, $0\le I\le u_{\max}/K_i$. Implement it as
`simulate_clamp(Kp, Ki, u_max)` and compare its peak output with plain PI and with conditional
integration, for $K_p=2$, $K_i=4$, $u_{\max}=1.2$. Which works better here, and why is neither perfect?

> 🤖 *Gemini tip:* "Compare integrator clamping and conditional integration as anti-windup methods for a PI controller with actuator saturation, and explain the trade-offs."

In [ ]:
# Your code here

---
## 8 · The D term: damping, and its price

A first-order plant does not need D. A **prosthetic elbow** does, because it has inertia. With joint
angle $\theta$, motor torque $u$, inertia $J$, a little joint friction $c$ and a load torque $\tau_L$ (the
forearm holding something):
$$J\ddot\theta=u-c\,\dot\theta-\tau_L .$$
Under P control, $J\ddot\theta+c\,\dot\theta+K_p\theta=K_p r$ — another mass–spring–damper (Section 7.1),
with damping ratio $\zeta=c/(2\sqrt{JK_p})$. The friction is tiny, so $\zeta\approx0.06$ and the arm
**rings**. The derivative term adds $-K_d\dot\theta$, which is pure extra damping:
$$\zeta=\frac{c+K_d}{2\sqrt{JK_p}} .$$
Holding a load, PD settles short by $\tau_L/K_p$ — the offset of Section 6 again — and a *small* I term
removes it. Two practical points are built into the code: D acts on the **measurement**
($-\dot\theta$) rather than on the error (Exercise below shows why), and the simulation is Euler's method
with a 1 ms step, well inside its stability limit.

In [ ]:
jt_J, jt_c = 0.06, 0.05          # kg m^2, N m s/rad  (forearm + prosthesis, light joint friction)
jt_dt, jt_T = 0.001, 4.0          # 1 kHz controller
jt_n = int(jt_T / jt_dt); jt_time = np.arange(jt_n) * jt_dt

def jt_sim(Kp, Ki=0.0, Kd=0.0, load=0.0, noise=0.0, d_on="measurement", tau_f=0.0, t_step=0.0, dt_j=jt_dt, seed=1):
    """Elbow under PID. Returns angle and torque. d_on: 'measurement' or 'error'; tau_f: derivative filter."""
    n_j = int(jt_T / dt_j); rng_j = np.random.default_rng(seed)
    th = w = I = d_f = 0.0; y_prev = e_prev = None; ths, us = [], []
    for k in range(n_j):
        r = 1.0 if k * dt_j >= t_step else 0.0          # setpoint: 1 rad (57 degrees)
        y = th + noise * rng_j.normal()                   # encoder reading
        e = r - y
        I += e * dt_j
        if d_on == "measurement":
            d_raw = 0.0 if y_prev is None else -(y - y_prev) / dt_j
        else:
            d_raw = 0.0 if e_prev is None else (e - e_prev) / dt_j
        d_f = d_raw if tau_f == 0 else d_f + dt_j / tau_f * (d_raw - d_f)   # first-order low-pass (an Euler step)
        u = Kp * e + Ki * I + Kd * d_f
        y_prev, e_prev = y, e
        th, w = th + dt_j * w, w + dt_j * (u - jt_c * w - load) / jt_J    # Euler on the two states
        ths.append(th); us.append(u)
    return np.array(ths), np.array(us)

jt_Kp = 3.0
zeta = lambda Kd: (jt_c + Kd) / (2 * np.sqrt(jt_J * jt_Kp))
jt_Kd = 0.8 * 2 * np.sqrt(jt_J * jt_Kp) - jt_c                  # choose Kd for zeta = 0.8
print(f"P only: zeta = {zeta(0):.3f}   |   Kd = {jt_Kd:.2f} gives zeta = {zeta(jt_Kd):.2f}")
runs = {"P": dict(), "PD": dict(Kd=jt_Kd), "PD, holding 0.5 N m": dict(Kd=jt_Kd, load=0.5),
        "PID, holding 0.5 N m": dict(Kd=jt_Kd, Ki=2.0, load=0.5)}
fig, ax = plt.subplots(figsize=(8, 4))
ax.axhline(1.0, ls="--", color="gray")
for (name, kw), col in zip(runs.items(), ["orange", "teal", "purple", "crimson"]):
    th_r, _ = jt_sim(jt_Kp, **kw)
    ax.plot(jt_time, th_r, color=col, label=name)
    print(f"  {name:22s} peak {th_r.max():.3f} rad, final {th_r[-1]:.3f} rad")
print(f"PD offset predicted by load/Kp: {0.5 / jt_Kp:.3f} rad")
ax.set_xlabel("time (s)"); ax.set_ylabel("elbow angle (rad)"); ax.legend(fontsize=8)
ax.set_title("P rings, D damps, I removes the load offset", fontsize=10); ax.grid(alpha=0.3); plt.show()

**Exercise 12.** **Derivative kick.** Let the setpoint step from 0 to 1 rad at $t=0.5$ s (`t_step=0.5`), with the elbow
at rest. Run the PD controller twice, with `d_on="error"` and with `d_on="measurement"`, and compare the
peak torque. (a) **Predict** the peak for the error version from $K_d\,\Delta e/\Delta t$ with $\Delta e=1$.
(b) Why are the two versions identical once the setpoint is constant? (c) What would a 600 N·m spike do to a
real prosthesis motor?

> 🤖 *Gemini tip:* "Explain derivative kick in PID controllers and why taking the derivative of the measurement instead of the error avoids it."

In [ ]:
# Your code here

### 8.1 · The derivative amplifies noise

Notebook 1 (Section 2) showed that a finite difference divides by $h$, so any error in the samples is
magnified by $1/h$. A controller is exactly that situation. With an encoder noise of $\sigma=0.002$ rad
(about 0.1°) and $\Delta t=1$ ms, the difference $(y_k-y_{k-1})/\Delta t$ carries noise of
$\sigma\sqrt2/\Delta t\approx2.8$ rad/s, and the D term turns it into **torque noise**
$K_d\,\sigma\sqrt2/\Delta t\approx1.8$ N·m. The angle barely notices; the motor chatters, heats and
wears.

The standard remedy is to **low-pass filter** the derivative,
$d_f\leftarrow d_f+\tfrac{\Delta t}{\tau_f}\,(d_{\text{raw}}-d_f)$ — itself an Euler step of
$\tau_f\dot d_f=d_{\text{raw}}-d_f$, i.e. a running weighted average (an integral!) of recent derivatives.

In [ ]:
jt_sigma = 0.002
pred_std = jt_Kd * np.sqrt(2) * jt_sigma / jt_dt
fig, ax = plt.subplots(figsize=(9, 3.6))
for tau_f, col, lw in [(0.0, "lightcoral", 0.5), (0.01, "darkorange", 1.2), (0.03, "teal", 1.6)]:
    th_n, u_n = jt_sim(jt_Kp, Kd=jt_Kd, noise=jt_sigma, tau_f=tau_f)
    late = jt_time > 2
    print(f"tau_f = {tau_f:4.2f} s: torque noise std {u_n[late].std():.3f} N m | angle noise std {th_n[late].std():.5f} rad")
    ax.plot(jt_time, u_n, color=col, lw=lw, label=f"τ_f = {tau_f} s" + ("  (raw derivative)" if tau_f == 0 else ""))
print(f"predicted for the raw derivative: Kd*sqrt(2)*sigma/dt = {pred_std:.3f} N m")
ax.set_xlim(1.5, 2.5); ax.set_ylim(-6, 6); ax.set_xlabel("time (s)"); ax.set_ylabel("motor torque (N m)")
ax.set_title("Same controller, same sensor: the raw derivative makes the motor chatter", fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.show()

**Exercise 13.** (a) The team wants to run the elbow controller ten times faster, at 10 kHz ($\Delta t=0.1$ ms), "for
better control". **Predict** the torque noise of the unfiltered D term, then **confirm** with
`jt_sim(..., dt_j=1e-4)`. (b) Back at 1 kHz, find the smallest filter constant $\tau_f$ among
`[0.005, 0.01, 0.02, 0.03]` that keeps the torque noise below 0.1 N·m, and check (without noise) that it
barely changes the step response's peak. (c) Why does sampling faster make a *derivative* worse but an
*integral* better?

> 🤖 *Gemini tip:* "Explain why the noise of a finite-difference derivative grows as the sampling interval shrinks, while numerical integration averages noise out, and how a first-order low-pass filter on the derivative term helps."

In [ ]:
# Your code here

---
### Module 3 complete
Integration as accumulation and area; Riemann sums and the Fundamental Theorem; numerical integration and
its error orders ($h$, $h^2$, $h^4$); a physiological flow → volume integral, and the drift that sensor bias
and noise build up; Euler's method as the integral behind every simulation, with the step-size law it shares
with gradient descent; and a PID controller from scratch — the integral that must drive the offset to zero,
tuning read off a characteristic equation, windup and anti-windup, and the derivative that damps a prosthetic
elbow at the price of amplifying noise. Differentiation amplifies noise, integration smooths it: the calculus
that links machine learning (Notebook 1: gradients) to control (this notebook: PID).
**Next:** Module 4 (OpenCV) applies the array, linear-algebra and calculus tools to biomedical images.